In [6]:
import rasterio
from rasterio.windows import Window
import os
from rasterio.features import bounds as feature_bounds
import geopandas as gpd
from shapely.geometry import box
from rasterio.crs import CRS
import fiona
from datetime import datetime
from tqdm import tqdm
import numpy as np
import json

Split geotiff to tiles

In [2]:
def split_geotiff(input_path, output_dir, tile_size=256):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    crs=''
    with rasterio.open(input_path) as src:
        # Get the dimensions of the input image
        width = src.width
        height = src.height

        num_tiles_x = width // tile_size
        num_tiles_y = height // tile_size

        for i in range(num_tiles_x):
            for j in range(num_tiles_y):
                window = Window(i * tile_size, j * tile_size, tile_size, tile_size)

                data = src.read(window=window)
                transform = src.window_transform(window)
                output_path = os.path.join(output_dir, f'tile_{i}_{j}.tif')

                with rasterio.open(
                    output_path, 'w', driver='GTiff',
                    height=tile_size, width=tile_size, count=src.count,
                    dtype=data.dtype, crs=src.crs, transform=transform,
                ) as dst:
                    dst.write(data)
    print(crs)
    print(f"Tiles have been saved to {output_dir}")

In [3]:
input_geotiff = 'data_split/20240720T072135_T40UFC_orig.tif'
output_directory = 'data_split/result_split'  # Directory to save the output tiles
split_geotiff(input_geotiff, output_directory)


Tiles have been saved to data_split/result_split


Split shp file to tiles by bounds from geotiff tiles

In [4]:
def get_geotiff_bounds(geotiff_path):
    with rasterio.open(geotiff_path) as src:
        #print(src.crs)
        return src.bounds

def clip_shapefile_by_bounds(shapefile_path, bounds, output_path):
    gdf = gpd.read_file(shapefile_path)
    #print(gdf.crs)
    bbox = box(*bounds)
    #print(bbox)
    #gdfbbox = box(*gdf)
    #print(gdf.bounds)
    clipped_gdf = gpd.clip(gdf, bbox)
    #clipped_gdf = gdf[gdf.intersects(bbox)]
    if not clipped_gdf.empty:
            clipped_gdf.to_file(output_path)
            print(f"Saved clipped shapefile: {output_path}")
    else:
            print(f"No features found within bounding box.")
    
    #clipped_gdf.to_file(output_path)

def process_geotiffs_and_shapefile(geotiff_folder, shapefile_path):
    for filename in os.listdir(geotiff_folder):
        if filename.endswith('.tif') or filename.endswith('.tiff'):
            geotiff_path = os.path.join(geotiff_folder, filename)
            bounds = get_geotiff_bounds(geotiff_path)
            output_shapefile_path = os.path.splitext(geotiff_path)[0] + '_clipped.shp'
            clip_shapefile_by_bounds(shapefile_path, bounds, output_shapefile_path)
            print(f"Clipped shapefile saved to {output_shapefile_path}")

In [5]:
geotiff_folder = 'data_split/result_split'
shapefile_path = 'data_split/vector_data.shp'
process_geotiffs_and_shapefile(geotiff_folder, shapefile_path)

No features found within bounding box.
Clipped shapefile saved to data_split/result_split\tile_0_0_clipped.shp
Saved clipped shapefile: data_split/result_split\tile_0_1_clipped.shp
Clipped shapefile saved to data_split/result_split\tile_0_1_clipped.shp
Saved clipped shapefile: data_split/result_split\tile_0_2_clipped.shp
Clipped shapefile saved to data_split/result_split\tile_0_2_clipped.shp
No features found within bounding box.
Clipped shapefile saved to data_split/result_split\tile_1_0_clipped.shp
Saved clipped shapefile: data_split/result_split\tile_1_1_clipped.shp
Clipped shapefile saved to data_split/result_split\tile_1_1_clipped.shp
Saved clipped shapefile: data_split/result_split\tile_1_2_clipped.shp
Clipped shapefile saved to data_split/result_split\tile_1_2_clipped.shp


Dataset to COCO

In [2]:
def create_coco_dataset(image_dir, shp_dir, output_json, categories):

    # Initialize COCO dataset structure
    coco_dataset = {
        "info": {
            "description": "Dataset created from TIFF images and SHP files",
            "version": "1.0",
            "year": datetime.now().year,
            "contributor": "",
            "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }

    # Add categories
    for idx, cat_name in enumerate(categories, 1):
        coco_dataset["categories"].append({
            "id": idx,
            "name": cat_name,
            "supercategory": "none"
        })

    # Get list of TIFF files
    tiff_files = [f for f in os.listdir(image_dir) if f.endswith(('.tif', '.tiff'))]
    
    image_id = 1
    annotation_id = 1

    # Process each TIFF file and its corresponding SHP file
    for tiff_file in tqdm(tiff_files, desc="Processing files"):
        base_name = os.path.splitext(tiff_file)[0]
        tiff_path = os.path.join(image_dir, tiff_file)
        shp_path = os.path.join(shp_dir, base_name + '_clipped.shp')

        if not os.path.exists(shp_path):
            print(f"Warning: No SHP file found for {tiff_file}")
            continue

        try:
            # Read image metadata
            with rasterio.open(tiff_path) as src:
                width = src.width
                height = src.height

            # Read SHP file
            gdf = gpd.read_file(shp_path)

            # Add image to COCO dataset
            coco_dataset["images"].append({
                "id": image_id,
                "file_name": tiff_file,
                "width": width,
                "height": height,
                "license": 0,
                "flickr_url": "",
                "coco_url": "",
                "date_captured": ""
            })

            # Process each geometry in the SHP file
            for idx, row in gdf.iterrows():
                geometry = row.geometry
                category_id = 1  # Default category ID (modify based on your data)

                # Handle different geometry types
                if geometry.type == 'Polygon':
                    polygons = [geometry]
                elif geometry.type == 'MultiPolygon':
                    polygons = list(geometry.geoms)
                else:
                    continue

                for polygon in polygons:
                    # Get exterior coordinates
                    exterior = np.array(polygon.exterior.coords)

                    # Convert to image coordinates (you might need to adjust this based on your CRS)
                    with rasterio.open(tiff_path) as src:
                        pixel_coords = [src.index(x, y) for x, y in exterior]
                        pixel_coords = [(y, x) for x, y in pixel_coords]  # Convert to (x, y)

                    # Flatten coordinates for COCO format
                    segmentation = []
                    for x, y in pixel_coords:
                        segmentation.extend([float(x), float(y)])

                    # Calculate bounding box [x, y, width, height]
                    x_coords = [p[0] for p in pixel_coords]
                    y_coords = [p[1] for p in pixel_coords]
                    x_min, x_max = min(x_coords), max(x_coords)
                    y_min, y_max = min(y_coords), max(y_coords)
                    bbox = [x_min, y_min, x_max - x_min, y_max - y_min]

                    # Calculate area
                    area = polygon.area

                    # Add annotation
                    coco_dataset["annotations"].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": category_id,
                        "segmentation": [segmentation],
                        "area": float(area),
                        "bbox": [float(x) for x in bbox],
                        "iscrowd": 0
                    })
                    annotation_id += 1

            image_id += 1

        except Exception as e:
            print(f"Error processing {tiff_file}: {str(e)}")
            continue

    # Save COCO dataset to JSON file
    with open(output_json, 'w') as f:
        json.dump(coco_dataset, f, indent=4)

    print(f"COCO dataset saved to {output_json}")
    print(f"Processed {image_id-1} images and {annotation_id-1} annotations")

In [7]:
image_dir = "data_split/result_split"  # Replace with your TIFF images directory
shp_dir = "data_split/result_split"      # Replace with your SHP files directory
output_json = "coco_dataset.json"  # Output COCO JSON file
categories = ["class1", "class2"]  # Replace with your category names

create_coco_dataset(image_dir, shp_dir, output_json, categories)

Processing files:   0%|                                                                          | 0/6 [00:00<?, ?it/s]C:\Users\alexe\AppData\Local\Temp\ipykernel_12556\3733559606.py:69: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'Polygon':


Processing files:  33%|██████████████████████                                            | 2/6 [00:00<00:00,  5.02it/s]C:\Users\alexe\AppData\Local\Temp\ipykernel_12556\3733559606.py:69: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'Polygon':
Processing files:  50%|█████████████████████████████████                                 | 3/6 [00:00<00:00,  3.88it/s]C:\Users\alexe\AppData\Local\Temp\ipykernel_12556\3733559606.py:69: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'Polygon':


Processing files:  83%|███████████████████████████████████████████████████████           | 5/6 [00:01<00:00,  5.05it/s]C:\Users\alexe\AppData\Local\Temp\ipykernel_12556\3733559606.py:69: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if geometry.type == 'Polygon':
Processing files: 100%|██████████████████████████████████████████████████████████████████| 6/6 [00:01<00:00,  4.55it/s]


COCO dataset saved to coco_dataset.json
Processed 4 images and 6 annotations
